In [ ]:
import shutil,subprocess,pathlib,os
r=subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader,nounits"],text=True,capture_output=True,check=True).stdout.strip()
name,mem=[x.strip() for x in r.splitlines()[0].rsplit(",",1)]
free=shutil.disk_usage("/content").free/1024**3
print(f"GPU: {name} | VRAM: {int(mem)/1024:.1f} GiB | Free disk: {free:.1f} GiB")
if int(mem)<24000: raise RuntimeError("Use a Colab GPU with >=24 GB VRAM for the supported pipeline.")
if free<30: raise RuntimeError("Need >=30 GiB free disk.")


In [ ]:
import pathlib,shutil
if pathlib.Path("/content/My-works").exists(): shutil.rmtree("/content/My-works")
!git clone -q --depth 1 https://github.com/Logan17de/My-works.git /content/My-works
TOOLS="/content/My-works/ai-3d-animation-engines/animation-engine"
OUTPUT_DIR="/content/animation_outputs"
pathlib.Path(OUTPUT_DIR).mkdir(parents=True,exist_ok=True)
!bash {TOOLS}/install_animation.sh


In [ ]:
import getpass,os,pathlib
from google.colab import files
token=getpass.getpass("Hugging Face token (Llama 3 access): ").strip()
if not token: raise ValueError("HF token required")
os.environ["HF_TOKEN"]=token
uploaded=files.upload()
if len(uploaded)!=1: raise ValueError("Upload exactly one humanoid GLB/FBX/OBJ/PLY")
target_name=next(iter(uploaded)); TARGET_CHARACTER=f"/content/{target_name}"
if pathlib.Path(TARGET_CHARACTER).suffix.lower() not in {".glb",".fbx",".obj",".ply"}: raise ValueError("Unsupported format")
PROMPT="A person walks forward, stops, and waves with the right hand."
DURATION_SECONDS=6.0; SEED=0; TARGET_ALREADY_RIGGED=False; MIA_NO_FINGERS=True


In [ ]:
import subprocess,shlex,os,pathlib,shutil
motion_stem=f"{OUTPUT_DIR}/motion"
cmd=["/opt/conda/bin/conda","run","-n","ardy","python","scripts/generate.py",PROMPT,"--model","core","--duration",str(DURATION_SECONDS),"--seed",str(SEED),"--output",motion_stem]
print("Running:"," ".join(shlex.quote(x) for x in cmd))
subprocess.run(cmd,cwd="/content/ardy",env=os.environ.copy(),check=True)
MOTION_NPZ=f"{motion_stem}.npz"; MOTION_BRIDGE=f"{OUTPUT_DIR}/motion_bridge.npz"; MOTION_PREVIEW=f"{OUTPUT_DIR}/motion_preview.mp4"
subprocess.run(["/opt/conda/bin/conda","run","-n","ardy","python",f"{TOOLS}/enrich_ardy_motion.py","--input",MOTION_NPZ,"--output",MOTION_BRIDGE],check=True)
subprocess.run(["/opt/conda/bin/conda","run","-n","ardy","python",f"{TOOLS}/preview_ardy_motion.py","--input",MOTION_BRIDGE,"--output",MOTION_PREVIEW],check=True)
from IPython.display import Video,display
display(Video(MOTION_PREVIEW,embed=True))
ARDY_SOURCE_FBX=f"{OUTPUT_DIR}/ardy_source.fbx"
subprocess.run(["/opt/conda/bin/conda","run","-n","mia","python",f"{TOOLS}/ardy_motion_to_fbx.py","--input",MOTION_BRIDGE,"--output",ARDY_SOURCE_FBX],check=True)

RIGGED_TARGET=f"{OUTPUT_DIR}/character_rigged.fbx"
if TARGET_ALREADY_RIGGED:
    if pathlib.Path(TARGET_CHARACTER).suffix.lower()!=".fbx": raise ValueError("Skip-rig requires FBX")
    shutil.copy2(TARGET_CHARACTER,RIGGED_TARGET)
else:
    c=["/opt/conda/bin/conda","run","-n","mia","python",f"{TOOLS}/rig_character_mia.py","--input",TARGET_CHARACTER,"--output",RIGGED_TARGET]
    if MIA_NO_FINGERS: c.append("--no-fingers")
    subprocess.run(c,env={**os.environ,"MIA_ROOT":"/content/Make-It-Animatable"},check=True)

FINAL_FBX=f"{OUTPUT_DIR}/character_animated.fbx"; FINAL_GLB=f"{OUTPUT_DIR}/character_animated.glb"
subprocess.run(["/opt/conda/bin/conda","run","-n","mia","python",f"{TOOLS}/retarget_with_mia.py","--target",RIGGED_TARGET,"--animation",ARDY_SOURCE_FBX,"--output",FINAL_FBX,"--preview-glb",FINAL_GLB],
               env={**os.environ,"MIA_ROOT":"/content/Make-It-Animatable"},check=True)
final=pathlib.Path(FINAL_FBX)
if not final.is_file() or final.stat().st_size==0: raise RuntimeError("Final FBX missing")
print(f"Final Unreal FBX: {FINAL_FBX} ({final.stat().st_size/1024**2:.1f} MiB)")


In [ ]:
from google.colab import files
files.download(FINAL_FBX)
